In [2]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---- ----------------------------------- 2.1/17.7 MB 13.1 MB/s eta 0:00:02
   ------------ --------------------------- 5.5/17.7 MB 15.9 MB/s eta 0:00:01
   ------------------------ --------------- 10.7/17.7 MB 19.3 MB/s eta 0:00:01
   ------------------------------------ --- 16.0/17.7 MB 20.9 MB/s eta 0:00:01
   ---------------------------------------- 17.7/17.7 MB 20.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import mysql.connector
from scipy import stats

# Connect and pull data
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Roshan@03',
    database='ecommerce_india'
)

df = pd.read_sql("SELECT * FROM vw_ecommerce_master", conn)
conn.close()

df.head()

C:\Users\Roshi\AppData\Local\Temp\ipykernel_26768\1371665815.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM vw_ecommerce_master", conn)


,Order ID,Order Date,CustomerName,State,City,Year,Month,month_year,Category,Sub-Category,Amount,Profit,Quantity,profit_margin_pct,price_band
0,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,2018,April,2018-04,Furniture,Bookcases,1275.0,-1148.0,7,-90.04,Premium
1,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,2018,April,2018-04,Clothing,Stole,66.0,-12.0,5,-18.18,Budget
2,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,2018,April,2018-04,Clothing,Hankerchief,8.0,-2.0,3,-25.00,Budget
3,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,2018,April,2018-04,Electronics,Electronic Games,80.0,-56.0,4,-70.00,Budget
4,B-25602,2018-04-01,Pearl,Maharashtra,Pune,2018,April,2018-04,Electronics,Phones,168.0,-111.0,2,-66.07,Budget


In [9]:
print(df['Category'].unique())
print(df['Category'].value_counts())

['Furniture' 'Clothing' 'Electronics']
Category
Clothing       2192
Electronics    1455
Furniture      1353
Name: count, dtype: int64


In [11]:
furniture_profit = df[df['Category']=='Furniture']['Profit']
clothing_profit = df[df['Category']=='Clothing']['Profit']    # capital C — this was the bug

t_stat, p_value = stats.ttest_ind(furniture_profit, clothing_profit, equal_var=False)

print(f"T-stats: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Statistically significant difference in profit between Furniture and Clothing")
else:
    print("No statistically significant difference - variation is likely due to chance")

T-stats: 3.445
P-value: 0.0006
Statistically significant difference in profit between Furniture and Clothing


### Interpretation:

T-statistic: 3.445, P-value: 0.0006
Since p < 0.05, there is a statistically significant difference in profit between Furniture and Clothing
The positive t-statistic means Furniture's profit is significantly higher than Clothing's (on average)

In [12]:
contingency_table = pd.crosstab(df['State'], df['price_band'])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2:.3f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Price band distribution is significantly associated with State (not random)")
else:
    print("No significant association — price band distribution is roughly the same across states")

Chi-square statistic: 346.454
P-value: 0.0000
Price band distribution is significantly associated with State (not random)


Significant result — chi-square = 346.454, p ≈ 0.0000 (well below 0.05).

### Interpretation:
There is a statistically significant association between State and price_band — meaning price-band preference genuinely varies by region, it's not random distribution. This directly supports the finding from your Page 2 dashboard where Premium dominated almost every state, but now you have statistical proof that the variation across states is real, not just visual noise.